Backward Propagation

In [1]:
import numpy as np

def sigmoid(z): return 1/(1+np.exp(-z))
def relu(z): return np.maximum(0, z)
def relu_derivative(z): return (z > 0).astype(float)

# Same network and input as worked example
x = np.array([[1],[2]])
y_true = 1

W1 = np.array([[0.5,-0.3],[0.2,0.8]])
b1 = np.array([[0.1],[-0.1]])
W2 = np.array([[0.6,-0.4]])
b2 = np.array([[0.2]])

lr = 0.1

# ---- FORWARD PASS ----
z1 = W1 @ x + b1
a1 = relu(z1)
z2 = W2 @ a1 + b2
a2 = sigmoid(z2)   # y_hat
y_hat = a2.item()
print("Prediction (y_hat):", y_hat)   # 0.382

loss = -(y_true*np.log(y_hat) + (1-y_true)*np.log(1-y_hat))
print("Loss:", loss)

# ---- BACKWARD PASS ----
delta2 = a2 - y_true              # simplified Sigmoid+BCE gradient
dW2 = delta2 @ a1.T
db2 = delta2

delta1 = (W2.T @ delta2) * relu_derivative(z1)
dW1 = delta1 @ x.T
db1 = delta1

print("delta2:", delta2.ravel())   # -0.618
print("dW2:", dW2.ravel())         # [0, -1.051]
print("dW1:\n", dW1)               # [[0,0],[0.247,0.494]]

# ---- UPDATE WEIGHTS ----
W2 -= lr * dW2
b2 -= lr * db2
W1 -= lr * dW1
b1 -= lr * db1

print("\nUpdated W2:", W2.ravel())   # [0.6, -0.295]
print("Updated b2:", b2.ravel())     # 0.262
print("Updated W1:\n", W1)

Prediction (y_hat): 0.382252125230751
Loss: 0.9616748743957433
delta2: [-0.61774787]
dW2: [-1.71459478e-17 -1.05017139e+00]
dW1:
 [[-0.37064872 -0.74129745]
 [ 0.24709915  0.4941983 ]]

Updated W2: [ 0.6        -0.29498286]
Updated b2: [0.26177479]
Updated W1:
 [[ 0.53706487 -0.22587026]
 [ 0.17529009  0.75058017]]


In [2]:
import numpy as np

class SimpleNeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, lr=0.1):
        np.random.seed(42)
        self.W1 = np.random.randn(hidden_size, input_size) * 0.1
        self.b1 = np.zeros((hidden_size, 1))
        self.W2 = np.random.randn(output_size, hidden_size) * 0.1
        self.b2 = np.zeros((output_size, 1))
        self.lr = lr

    def relu(self, z): return np.maximum(0, z)
    def relu_derivative(self, z): return (z > 0).astype(float)
    def sigmoid(self, z): return 1/(1+np.exp(-z))

    def forward(self, x):
        self.z1 = self.W1 @ x + self.b1
        self.a1 = self.relu(self.z1)
        self.z2 = self.W2 @ self.a1 + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2

    def backward(self, x, y_true):
        m = x.shape[1]  # number of samples in this batch
        delta2 = self.a2 - y_true
        dW2 = (delta2 @ self.a1.T) / m
        db2 = np.sum(delta2, axis=1, keepdims=True) / m

        delta1 = (self.W2.T @ delta2) * self.relu_derivative(self.z1)
        dW1 = (delta1 @ x.T) / m
        db1 = np.sum(delta1, axis=1, keepdims=True) / m

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def compute_loss(self, y_true, y_pred, eps=1e-15):
        y_pred = np.clip(y_pred, eps, 1-eps)
        return -np.mean(y_true*np.log(y_pred) + (1-y_true)*np.log(1-y_pred))

    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss = self.compute_loss(y, y_pred)
            self.backward(X, y)
            if epoch % 100 == 0:
                print(f"Epoch {epoch}: Loss = {loss:.4f}")

# XOR problem — the classic example a single perceptron CANNOT solve, but a hidden layer CAN
X = np.array([[0,0,1,1],[0,1,0,1]])   # 2 features, 4 samples (columns)
y = np.array([[0,1,1,0]])              # XOR labels

nn = SimpleNeuralNetwork(input_size=2, hidden_size=4, output_size=1, lr=0.5)
nn.train(X, y, epochs=2000)

predictions = nn.forward(X)
print("\nFinal predictions:", predictions.round(3))
print("Rounded to classes:", (predictions > 0.5).astype(int))
print("True labels:", y)

Epoch 0: Loss = 0.6932
Epoch 100: Loss = 0.5690
Epoch 200: Loss = 0.1544
Epoch 300: Loss = 0.0454
Epoch 400: Loss = 0.0236
Epoch 500: Loss = 0.0155
Epoch 600: Loss = 0.0114
Epoch 700: Loss = 0.0089
Epoch 800: Loss = 0.0073
Epoch 900: Loss = 0.0061
Epoch 1000: Loss = 0.0053
Epoch 1100: Loss = 0.0046
Epoch 1200: Loss = 0.0041
Epoch 1300: Loss = 0.0037
Epoch 1400: Loss = 0.0034
Epoch 1500: Loss = 0.0031
Epoch 1600: Loss = 0.0028
Epoch 1700: Loss = 0.0026
Epoch 1800: Loss = 0.0025
Epoch 1900: Loss = 0.0023

Final predictions: [[0.006 0.999 0.999 0.001]]
Rounded to classes: [[0 1 1 0]]
True labels: [[0 1 1 0]]


In [3]:
import numpy as np

# 1. Activation Function & Derivative
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_derivative(out):
    # Derivative of Sigmoid given its output value: σ'(z) = σ(z) * (1 - σ(z))
    return out * (1.0 - out)

# 2. Inputs, Targets, and Hyperparameters
i1, i2 = 0.05, 0.10
target_o1, target_o2 = 0.01, 0.99
eta = 0.5  # Learning rate

b1, b2 = 0.35, 0.60

# Initial Weights
w1, w2, w3, w4 = 0.15, 0.20, 0.25, 0.30
w5, w6, w7, w8 = 0.40, 0.45, 0.50, 0.55

# ==========================================
# PHASE 1: FORWARD PROPAGATION
# ==========================================
# Hidden Layer
net_h1 = (i1 * w1) + (i2 * w2) + b1
out_h1 = sigmoid(net_h1)

net_h2 = (i1 * w3) + (i2 * w4) + b1
out_h2 = sigmoid(net_h2)

# Output Layer
net_o1 = (out_h1 * w5) + (out_h2 * w6) + b2
out_o1 = sigmoid(net_o1)

net_o2 = (out_h1 * w7) + (out_h2 * w8) + b2
out_o2 = sigmoid(net_o2)

# Total Loss
e_o1 = 0.5 * (target_o1 - out_o1) ** 2
e_o2 = 0.5 * (target_o2 - out_o2) ** 2
e_total = e_o1 + e_o2

# ==========================================
# PHASE 2: BACKWARD PROPAGATION
# ==========================================

# 1. Output Layer Gradients (deltas)
# delta_o = dE/d_out * d_out/d_net
delta_o1 = -(target_o1 - out_o1) * sigmoid_derivative(out_o1)
delta_o2 = -(target_o2 - out_o2) * sigmoid_derivative(out_o2)

# Gradients w.r.t Output Weights
dE_dw5 = delta_o1 * out_h1
dE_dw6 = delta_o1 * out_h2
dE_dw7 = delta_o2 * out_h1
dE_dw8 = delta_o2 * out_h2

# 2. Hidden Layer Gradients (deltas)
# Sum error contributions from output layer
dE_dout_h1 = (delta_o1 * w5) + (delta_o2 * w7)
dE_dout_h2 = (delta_o1 * w6) + (delta_o2 * w8)

delta_h1 = dE_dout_h1 * sigmoid_derivative(out_h1)
delta_h2 = dE_dout_h2 * sigmoid_derivative(out_h2)

# Gradients w.r.t Hidden Weights
dE_dw1 = delta_h1 * i1
dE_dw2 = delta_h1 * i2
dE_dw3 = delta_h2 * i1
dE_dw4 = delta_h2 * i2

# 3. Update Weights
w1_new = w1 - (eta * dE_dw1)
w2_new = w2 - (eta * dE_dw2)
w3_new = w3 - (eta * dE_dw3)
w4_new = w4 - (eta * dE_dw4)

w5_new = w5 - (eta * dE_dw5)
w6_new = w6 - (eta * dE_dw6)
w7_new = w7 - (eta * dE_dw7)
w8_new = w8 - (eta * dE_dw8)

# ==========================================
# DISPLAY RESULTS
# ==========================================
print("--- FORWARD PASS RESULTS ---")
print(f"out_h1 : {out_h1:.7f}")
print(f"out_h2 : {out_h2:.7f}")
print(f"out_o1 : {out_o1:.7f}")
print(f"out_o2 : {out_o2:.7f}")
print(f"Loss   : {e_total:.7f}\n")

print("--- BACKWARD PASS GRADIENTS ---")
print(f"dE/dw5 : {dE_dw5:.7f}")
print(f"dE/dw6 : {dE_dw6:.7f}")
print(f"dE/dw7 : {dE_dw7:.7f}")
print(f"dE/dw8 : {dE_dw8:.7f}")
print(f"dE/dw1 : {dE_dw1:.7f}")
print(f"dE/dw2 : {dE_dw2:.7f}")
print(f"dE/dw3 : {dE_dw3:.7f}")
print(f"dE/dw4 : {dE_dw4:.7f}\n")

print("--- UPDATED WEIGHTS (eta = 0.5) ---")
print(f"w1+ : {w1_new:.7f} | w2+ : {w2_new:.7f}")
print(f"w3+ : {w3_new:.7f} | w4+ : {w4_new:.7f}")
print(f"w5+ : {w5_new:.7f} | w6+ : {w6_new:.7f}")
print(f"w7+ : {w7_new:.7f} | w8+ : {w8_new:.7f}")

--- FORWARD PASS RESULTS ---
out_h1 : 0.5932700
out_h2 : 0.5968844
out_o1 : 0.7513651
out_o2 : 0.7729285
Loss   : 0.2983711

--- BACKWARD PASS GRADIENTS ---
dE/dw5 : 0.0821670
dE/dw6 : 0.0826676
dE/dw7 : -0.0226025
dE/dw8 : -0.0227402
dE/dw1 : 0.0004386
dE/dw2 : 0.0008771
dE/dw3 : 0.0004977
dE/dw4 : 0.0009954

--- UPDATED WEIGHTS (eta = 0.5) ---
w1+ : 0.1497807 | w2+ : 0.1995614
w3+ : 0.2497511 | w4+ : 0.2995023
w5+ : 0.3589165 | w6+ : 0.4086662
w7+ : 0.5113013 | w8+ : 0.5613701


Optimizers

In [4]:
# pip install tensorflow --break-system-packages
import tensorflow as tf
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Dense(4, activation='relu', input_shape=(2,)),
    keras.layers.Dense(1, activation='sigmoid')
])

# Different optimizer choices — this is where all the math above gets applied automatically
model.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])
# model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1, momentum=0.9), loss='binary_crossentropy')
# model.compile(optimizer='rmsprop', loss='binary_crossentropy')
# model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy')

import numpy as np
X = np.array([[0,0],[0,1],[1,0],[1,1]])
y = np.array([0,1,1,0])

history = model.fit(X, y, epochs=200, verbose=0)
print("Final loss:", history.history['loss'][-1])
print("Predictions:", model.predict(X).round(3))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Final loss: 0.6931471824645996
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Predictions: [[0.5]
 [0.5]
 [0.5]
 [0.5]]
